# `06-deepagent/test.ipynb`
1. csv 파일을 받아서
2. 데이터 분석을 진행하고
3. 슬랙으로 결과를 보냄

```sh
uv pip install deepagents slack-sdk langcahain-daytona langchain-openai
```

In [ ]:
from dotenv import load_dotenv

load_dotenv()

### Daytona Sandbox

In [ ]:
from daytona import Daytona
from langchain_daytona import DaytonaSandbox

sandbox = Daytona().create()
backend = DaytonaSandbox(sandbox=sandbox)

In [ ]:
result = backend.execute('ls -a')
print(result.output)

In [ ]:
import csv
import io

data = [
    ["Date", "Product", "Units Sold", "Revenue"],
    ["2025-08-01", "Widget A", 10, 250],
    ["2025-08-02", "Widget B", 5, 125],
    ["2025-08-03", "Widget A", 7, 175],
    ["2025-08-04", "Widget C", 3, 90],
    ["2025-08-05", "Widget B", 8, 200],
]

buf = io.StringIO()
writer = csv.writer(buf)
writer.writerows(data)

# 원본 csv 데이터
csv_bytes = buf.getvalue().encode('utf-8')
buf.close()

# 파일 저장(지금 우리는 필요 없음)
with open('./sales.csv', 'wb') as f:
    f.write(csv_bytes)

# sandbox 에 업로드
backend.upload_files(
    [
        ('/home/daytona/data/sales_data.csv', csv_bytes)
    ]
)


### Slack
- Agent 가 사용할 Slack Messaging Tool 생성

In [ ]:
import os
from slack_sdk import WebClient

SLACK_BOT_TOKEN = os.getenv('SLACK_BOT_TOKEN')

client = WebClient(token=SLACK_BOT_TOKEN)

# 봇이 접근 가능한 모든 채널 리스트
res = client.conversations_list()
for ch in res['channels']:
    print(ch['name'], ch['id'])

social_channel_id = 'C0A3WG6GLQK'

client.chat_postMessage(
    channel=social_channel_id,
    text='hello'
)

In [ ]:
import os
from slack_sdk import WebClient
from langchain.tools import tool

SLACK_TOKEN = os.getenv('SLACK_BOT_TOKEN')
slack_client = WebClient(token=SLACK_TOKEN)


@tool(parse_docstring=True)  # LLM에게 Tool 설명을 docstring 에서 제공
def send_slack_message(text: str, file_path: str | None = None) -> str:
    """메세지 전송, 경우에 따라 이미지같은 파일을 첨부함
    
    Args:
        text: (str) 메세지의 내용
        file_path: (str) 파일시스템 상 첨부파일의 경로
    """

    social_channel_id = 'C0A3WG6GLQK'
    # 첨부 파일 없으면
    if not file_path:
        slack_client.chat_postMessage(channel=social_channel_id, text=text)
    else:
        fp = backend.download_files([file_path])
        slack_client.files_upload_v2(
            channel=social_channel_id,
            content=fp[0].content,
            initial_comment=text,
        )
    
    return 'Message sent.'

## Build Deep Agent

In [ ]:
import uuid

from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent

checkpointer = InMemorySaver()

agent = create_deep_agent(
    model='openai:gpt-5-mini',
    tools=[send_slack_message],
    backend=backend,
    checkpointer=checkpointer,
)

thread_id = str(uuid.uuid4())
config = {'configurable': {'thread_id': thread_id}}

In [ ]:
input_message = {
    'role': 'user',
    'content': '''
    현재 폴더 안에 ./data/sales_data.csv 파일을 분석하고, 시각화 해줘.
    다 끝나면 분석결과와 시각화 이미지를 Slack 메세지로 보내줘.
    '''
}

for step in agent.stream(
    {"messages": [input_message]},
    config,
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

In [ ]:
input_message = {
    'role': 'user',
    'content': '''
    채널은 기본 세팅 되어있음. 2번 데이터 시각화 해서 슬랙 메시지로 보내줘
    '''
}

for step in agent.stream(
    {"messages": [input_message]},
    config,
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

In [60]:
from daytona import Daytona


Daytona().find_one('68a5645c-92f7-4324-aa7e-c24467cf5e35')

Daytona().list().items[0]

Sandbox(id='68a5645c-92f7-4324-aa7e-c24467cf5e35', organization_id='e4d2b24e-6f6e-4b5b-b55f-f61865b1d48e', name='68a5645c-92f7-4324-aa7e-c24467cf5e35', snapshot='daytonaio/sandbox:0.5.1', user='daytona', env={}, labels={}, public=False, network_block_all=False, network_allow_list=None, target='us', cpu=1, gpu=0, memory=1, disk=3, state=<SandboxState.STARTED: 'started'>, desired_state=<SandboxDesiredState.STARTED: 'started'>, error_reason=None, recoverable=False, backup_state='Completed', backup_created_at='2026-03-10T06:55:33.335Z', auto_stop_interval=15, auto_archive_interval=10080, auto_delete_interval=-1, volumes=[], build_info=None, created_at='2026-03-10T06:51:52.149Z', updated_at='2026-03-10T06:55:33.742Z', var_class='small', daemon_version='v0.149.0-prod', runner_id='5bb05fd5-e3e4-4053-9848-b39297a773a4', toolbox_proxy_url='https://proxy.app.daytona.io/toolbox', additional_properties={})